# Criação da camada Silver

## Objetivo

Este notebook transforma os dados brutos da camada Bronze em estruturas tratadas e confiáveis para consumo analítico.

Nesta etapa são realizados:

- padronização de textos;
- tipagem de colunas;
- tratamento de valores nulos;
- remoção de duplicidades;
- aplicação de regras básicas de qualidade;
- identificação de registros inválidos;
- organização dos dados em dimensões e fatos.

## Estrutura

As tabelas Silver são organizadas em:

### Dimensões

- `dim_cliente`;
- `dim_produto`;
- `dim_vendedor`;
- `dim_fornecedor`.

### Fatos

- `fato_vendas`;
- `fato_compras`;
- `fato_estoque`;
- `fato_contas_receber`;
- `fato_contas_pagar`;
- `fato_metas_vendas`.

### Quarentena

Registros que violam regras essenciais de qualidade não são descartados silenciosamente.

Quando necessário, eles são armazenados em tabelas de quarentena para permitir rastreabilidade.

## Fluxo

Bronze → Tratamento → Silver

A camada Silver preserva a granularidade necessária para as análises, enquanto disponibiliza dados padronizados e confiáveis para a construção da camada Gold.

In [0]:
from pyspark.sql import functions as F


# ---------------------------------------------------------
# CONFIGURAÇÃO
# ---------------------------------------------------------

catalogo_atual = spark.sql(
    "SELECT current_catalog()"
).first()[0]


schema_bronze = "varejo_bronze"
schema_silver = "varejo_silver"


print(f"Catálogo: {catalogo_atual}")
print(f"Origem: {schema_bronze}")
print(f"Destino: {schema_silver}")

In [0]:
# ---------------------------------------------------------
# FUNÇÕES AUXILIARES
# ---------------------------------------------------------

def carregar_bronze(nome_tabela):
    """
    Carrega uma tabela da camada Bronze.
    """

    return spark.table(
        f"{catalogo_atual}."
        f"{schema_bronze}."
        f"{nome_tabela}"
    )


def salvar_silver(
    dataframe,
    nome_tabela
):
    """
    Grava um DataFrame como tabela Delta
    gerenciada na camada Silver.
    """

    nome_completo = (
        f"{catalogo_atual}."
        f"{schema_silver}."
        f"{nome_tabela}"
    )


    (
        dataframe
        .write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true"
        )
        .saveAsTable(
            nome_completo
        )
    )


    print(
        f"Tabela criada: {nome_completo}"
    )

In [0]:
# ---------------------------------------------------------
# DIMENSÃO CLIENTE
# ---------------------------------------------------------

df_clientes_bronze = carregar_bronze(
    "clientes"
)


df_dim_cliente = (

    df_clientes_bronze

    .select(

        F.upper(
            F.trim(
                F.col("id_cliente")
            )
        ).alias(
            "id_cliente"
        ),

        F.trim(
            F.col("nome_cliente")
        ).alias(
            "nome_cliente"
        ),

        F.when(
            F.col(
                "segmento_cliente"
            ).isNull()
            |
            (
                F.trim(
                    F.col(
                        "segmento_cliente"
                    )
                ) == ""
            ),

            F.lit(
                "Não informado"
            )

        ).otherwise(

            F.trim(
                F.col(
                    "segmento_cliente"
                )
            )

        ).alias(
            "segmento_cliente"
        ),

        F.trim(
            F.col("cidade")
        ).alias(
            "cidade"
        ),

        F.upper(
            F.trim(
                F.col("estado")
            )
        ).alias(
            "estado"
        ),

        F.to_date(
            F.col("data_cadastro")
        ).alias(
            "data_cadastro"
        ),

        F.trim(
            F.col("porte_cliente")
        ).alias(
            "porte_cliente"
        ),

        F.col(
            "limite_credito"
        )
        .cast(
            "decimal(18,2)"
        )
        .alias(
            "limite_credito"
        ),

        F.trim(
            F.col(
                "situacao_cliente"
            )
        ).alias(
            "situacao_cliente"
        ),

        F.col(
            "_data_ingestao"
        ).alias(
            "_data_ingestao_origem"
        )
    )

    .filter(
        F.col(
            "id_cliente"
        ).isNotNull()
    )

    .dropDuplicates(
        ["id_cliente"]
    )

    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)

In [0]:
salvar_silver(
    df_dim_cliente,
    "dim_cliente"
)

In [0]:
display(
    df_dim_cliente.limit(20)
)

In [0]:
df_dim_cliente.printSchema()

In [0]:
# ---------------------------------------------------------
# DIMENSÃO PRODUTO
# ---------------------------------------------------------

df_produtos_bronze = carregar_bronze(
    "produtos"
)


df_dim_produto = (

    df_produtos_bronze

    .select(

        F.upper(
            F.trim(
                F.col("id_produto")
            )
        ).alias(
            "id_produto"
        ),

        F.trim(
            F.col("nome_produto")
        ).alias(
            "nome_produto"
        ),

        F.trim(
            F.col("categoria")
        ).alias(
            "categoria"
        ),

        F.trim(
            F.col("subcategoria")
        ).alias(
            "subcategoria"
        ),

        F.trim(
            F.col("marca")
        ).alias(
            "marca"
        ),

        F.upper(
            F.trim(
                F.col("unidade_medida")
            )
        ).alias(
            "unidade_medida"
        ),

        F.col(
            "custo_unitario"
        )
        .cast(
            "decimal(18,2)"
        )
        .alias(
            "custo_unitario"
        ),

        F.col(
            "preco_venda"
        )
        .cast(
            "decimal(18,2)"
        )
        .alias(
            "preco_venda"
        ),

        F.col(
            "margem_padrao"
        )
        .cast(
            "decimal(10,4)"
        )
        .alias(
            "margem_padrao"
        ),

        F.upper(
            F.trim(
                F.col(
                    "id_fornecedor_principal"
                )
            )
        ).alias(
            "id_fornecedor_principal"
        ),

        F.trim(
            F.col(
                "situacao_produto"
            )
        ).alias(
            "situacao_produto"
        ),

        F.col(
            "_data_ingestao"
        ).alias(
            "_data_ingestao_origem"
        )
    )

    .filter(
        F.col(
            "id_produto"
        ).isNotNull()
    )

    .dropDuplicates(
        ["id_produto"]
    )

    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)


salvar_silver(
    df_dim_produto,
    "dim_produto"
)

In [0]:
# ---------------------------------------------------------
# DIMENSÃO VENDEDOR
# ---------------------------------------------------------

df_vendedores_bronze = carregar_bronze(
    "vendedores"
)


df_dim_vendedor = (

    df_vendedores_bronze

    .select(

        F.upper(
            F.trim(
                F.col("id_vendedor")
            )
        ).alias(
            "id_vendedor"
        ),

        F.trim(
            F.col("nome_vendedor")
        ).alias(
            "nome_vendedor"
        ),

        F.to_date(
            F.col("data_admissao")
        ).alias(
            "data_admissao"
        ),

        F.trim(
            F.col("regiao")
        ).alias(
            "regiao"
        ),

        F.trim(
            F.col("nivel")
        ).alias(
            "nivel"
        ),

        F.trim(
            F.col(
                "situacao_vendedor"
            )
        ).alias(
            "situacao_vendedor"
        ),

        F.col(
            "_data_ingestao"
        ).alias(
            "_data_ingestao_origem"
        )
    )

    .dropDuplicates(
        ["id_vendedor"]
    )

    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)


salvar_silver(
    df_dim_vendedor,
    "dim_vendedor"
)

In [0]:
# ---------------------------------------------------------
# DIMENSÃO FORNECEDOR
# ---------------------------------------------------------

df_fornecedores_bronze = carregar_bronze(
    "fornecedores"
)


df_dim_fornecedor = (

    df_fornecedores_bronze

    .select(

        F.upper(
            F.trim(
                F.col("id_fornecedor")
            )
        ).alias(
            "id_fornecedor"
        ),

        F.trim(
            F.col("nome_fornecedor")
        ).alias(
            "nome_fornecedor"
        ),

        F.upper(
            F.trim(
                F.col("estado")
            )
        ).alias(
            "estado"
        ),

        F.col(
            "prazo_medio_entrega"
        )
        .cast("int")
        .alias(
            "prazo_medio_entrega"
        ),

        F.trim(
            F.col(
                "condicao_pagamento"
            )
        ).alias(
            "condicao_pagamento"
        ),

        F.col(
            "avaliacao_fornecedor"
        )
        .cast(
            "decimal(5,2)"
        )
        .alias(
            "avaliacao_fornecedor"
        ),

        F.trim(
            F.col(
                "situacao_fornecedor"
            )
        ).alias(
            "situacao_fornecedor"
        ),

        F.col(
            "_data_ingestao"
        ).alias(
            "_data_ingestao_origem"
        )
    )

    .dropDuplicates(
        ["id_fornecedor"]
    )

    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)


salvar_silver(
    df_dim_fornecedor,
    "dim_fornecedor"
)

In [0]:
# ---------------------------------------------------------
# FATO VENDAS
# ---------------------------------------------------------

df_vendas_bronze = carregar_bronze(
    "vendas"
)


df_vendas_preparadas = (

    df_vendas_bronze

    .select(

        F.upper(
            F.trim(
                F.col("id_item_venda")
            )
        ).alias(
            "id_item_venda"
        ),

        F.upper(
            F.trim(
                F.col("id_venda")
            )
        ).alias(
            "id_venda"
        ),

        F.to_date(
            F.col("data_venda")
        ).alias(
            "data_venda"
        ),

        F.upper(
            F.trim(
                F.col("id_cliente")
            )
        ).alias(
            "id_cliente"
        ),

        F.upper(
            F.trim(
                F.col("id_produto")
            )
        ).alias(
            "id_produto"
        ),

        F.upper(
            F.trim(
                F.col("id_vendedor")
            )
        ).alias(
            "id_vendedor"
        ),

        F.col(
            "quantidade"
        )
        .cast("int")
        .alias(
            "quantidade"
        ),

        F.col(
            "preco_unitario"
        )
        .cast(
            "decimal(18,2)"
        )
        .alias(
            "preco_unitario"
        ),

        F.col(
            "valor_bruto"
        )
        .cast(
            "decimal(18,2)"
        )
        .alias(
            "valor_bruto"
        ),

        F.col(
            "percentual_desconto"
        )
        .cast(
            "decimal(10,4)"
        )
        .alias(
            "percentual_desconto"
        ),

        F.col(
            "valor_desconto"
        )
        .cast(
            "decimal(18,2)"
        )
        .alias(
            "valor_desconto"
        ),

        F.col(
            "valor_liquido"
        )
        .cast(
            "decimal(18,2)"
        )
        .alias(
            "valor_liquido"
        ),

        F.col(
            "custo_total"
        )
        .cast(
            "decimal(18,2)"
        )
        .alias(
            "custo_total"
        ),

        F.col(
            "lucro_bruto"
        )
        .cast(
            "decimal(18,2)"
        )
        .alias(
            "lucro_bruto"
        ),

        F.col(
            "margem_percentual"
        )
        .cast(
            "decimal(10,4)"
        )
        .alias(
            "margem_percentual"
        ),

        F.col(
            "_data_ingestao"
        ).alias(
            "_data_ingestao_origem"
        )
    )

    # Remove as duplicidades introduzidas
    # propositalmente na origem.

    .dropDuplicates(
        ["id_item_venda"]
    )
)

In [0]:
df_vendas_avaliadas = (

    df_vendas_preparadas

    .withColumn(

        "motivo_rejeicao",

        F.when(
            F.col(
                "id_item_venda"
            ).isNull(),

            F.lit(
                "Identificador do item não informado"
            )
        )

        .when(
            F.col(
                "id_venda"
            ).isNull(),

            F.lit(
                "Identificador da venda não informado"
            )
        )

        .when(
            F.col(
                "id_cliente"
            ).isNull(),

            F.lit(
                "Cliente não informado"
            )
        )

        .when(
            F.col(
                "id_produto"
            ).isNull(),

            F.lit(
                "Produto não informado"
            )
        )

        .when(
            F.col(
                "id_vendedor"
            ).isNull(),

            F.lit(
                "Vendedor não informado"
            )
        )

        .when(
            F.col(
                "data_venda"
            ).isNull(),

            F.lit(
                "Data da venda inválida"
            )
        )

        .when(
            F.col(
                "quantidade"
            ) <= 0,

            F.lit(
                "Quantidade inválida"
            )
        )

        .when(
            F.col(
                "valor_liquido"
            ) < 0,

            F.lit(
                "Valor líquido inválido"
            )
        )
    )
)

In [0]:
# Registros que apresentam problema

df_quarentena_vendas = (

    df_vendas_avaliadas

    .filter(
        F.col(
            "motivo_rejeicao"
        ).isNotNull()
    )

    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)


salvar_silver(
    df_quarentena_vendas,
    "quarentena_vendas"
)

In [0]:
display(
    df_quarentena_vendas
    .groupBy(
        "motivo_rejeicao"
    )
    .count()
)

In [0]:
df_fato_vendas = (

    df_vendas_avaliadas

    .filter(
        F.col(
            "motivo_rejeicao"
        ).isNull()
    )

    .drop(
        "motivo_rejeicao"
    )

    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)


salvar_silver(
    df_fato_vendas,
    "fato_vendas"
)

In [0]:
# ---------------------------------------------------------
# FATO COMPRAS
# ---------------------------------------------------------

df_compras_bronze = carregar_bronze(
    "compras"
)


df_fato_compras = (

    df_compras_bronze

    .select(

        F.upper(
            F.trim(
                F.col("id_item_compra")
            )
        ).alias(
            "id_item_compra"
        ),

        F.upper(
            F.trim(
                F.col("id_compra")
            )
        ).alias(
            "id_compra"
        ),

        F.to_date(
            F.col("data_compra")
        ).alias(
            "data_compra"
        ),

        F.upper(
            F.trim(
                F.col("id_fornecedor")
            )
        ).alias(
            "id_fornecedor"
        ),

        F.upper(
            F.trim(
                F.col("id_produto")
            )
        ).alias(
            "id_produto"
        ),

        F.col(
            "quantidade"
        )
        .cast("int")
        .alias(
            "quantidade"
        ),

        F.col(
            "custo_unitario"
        )
        .cast(
            "decimal(18,2)"
        )
        .alias(
            "custo_unitario"
        ),

        F.col(
            "valor_total"
        )
        .cast(
            "decimal(18,2)"
        )
        .alias(
            "valor_total"
        ),

        F.col(
            "prazo_entrega"
        )
        .cast("int")
        .alias(
            "prazo_entrega"
        ),

        F.to_date(
            F.col(
                "data_recebimento"
            )
        ).alias(
            "data_recebimento"
        ),

        F.col(
            "_data_ingestao"
        ).alias(
            "_data_ingestao_origem"
        )
    )

    .dropDuplicates(
        ["id_item_compra"]
    )

    .filter(
        F.col(
            "quantidade"
        ) > 0
    )

    .filter(
        F.col(
            "valor_total"
        ) >= 0
    )

    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)


salvar_silver(
    df_fato_compras,
    "fato_compras"
)

In [0]:
# ---------------------------------------------------------
# FATO ESTOQUE
# ---------------------------------------------------------

df_estoque_bronze = carregar_bronze(
    "estoque"
)


df_fato_estoque = (

    df_estoque_bronze

    .select(

        F.to_date(
            F.col(
                "data_referencia"
            )
        ).alias(
            "data_referencia"
        ),

        F.upper(
            F.trim(
                F.col("id_produto")
            )
        ).alias(
            "id_produto"
        ),

        F.col(
            "quantidade_estoque"
        )
        .cast("int")
        .alias(
            "quantidade_estoque"
        ),

        F.col(
            "custo_medio"
        )
        .cast(
            "decimal(18,2)"
        )
        .alias(
            "custo_medio"
        ),

        F.col(
            "valor_estoque"
        )
        .cast(
            "decimal(18,2)"
        )
        .alias(
            "valor_estoque"
        ),

        F.col(
            "estoque_minimo"
        )
        .cast("int")
        .alias(
            "estoque_minimo"
        ),

        F.col(
            "estoque_maximo"
        )
        .cast("int")
        .alias(
            "estoque_maximo"
        ),

        F.col(
            "_data_ingestao"
        ).alias(
            "_data_ingestao_origem"
        )
    )

    .dropDuplicates(
        [
            "data_referencia",
            "id_produto"
        ]
    )

    .filter(
        F.col(
            "quantidade_estoque"
        ) >= 0
    )

    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)


salvar_silver(
    df_fato_estoque,
    "fato_estoque"
)

In [0]:
# ---------------------------------------------------------
# FATO CONTAS A RECEBER
# ---------------------------------------------------------

df_receber_bronze = carregar_bronze(
    "contas_receber"
)


df_fato_contas_receber = (

    df_receber_bronze

    .select(

        F.upper(
            F.trim(
                F.col(
                    "id_titulo_receber"
                )
            )
        ).alias(
            "id_titulo_receber"
        ),

        F.upper(
            F.trim(
                F.col("id_venda")
            )
        ).alias(
            "id_venda"
        ),

        F.upper(
            F.trim(
                F.col("id_cliente")
            )
        ).alias(
            "id_cliente"
        ),

        F.to_date(
            F.col("data_emissao")
        ).alias(
            "data_emissao"
        ),

        F.to_date(
            F.col("data_vencimento")
        ).alias(
            "data_vencimento"
        ),

        F.to_date(
            F.col("data_pagamento")
        ).alias(
            "data_pagamento"
        ),

        F.col(
            "valor_titulo"
        )
        .cast(
            "decimal(18,2)"
        )
        .alias(
            "valor_titulo"
        ),

        F.trim(
            F.col("status_titulo")
        ).alias(
            "status_titulo"
        ),

        F.col(
            "dias_atraso"
        )
        .cast("int")
        .alias(
            "dias_atraso"
        ),

        F.col(
            "_data_ingestao"
        ).alias(
            "_data_ingestao_origem"
        )
    )

    .dropDuplicates(
        ["id_titulo_receber"]
    )

    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)


salvar_silver(
    df_fato_contas_receber,
    "fato_contas_receber"
)

In [0]:
# ---------------------------------------------------------
# INTEGRIDADE ENTRE CONTAS A RECEBER E VENDAS
# ---------------------------------------------------------

df_vendas_validas = (

    df_fato_vendas

    .select(
        "id_venda"
    )

    .distinct()
)

In [0]:
df_quarentena_contas_receber = (

    df_fato_contas_receber.alias("r")

    .join(
        df_vendas_validas.alias("v"),

        on="id_venda",

        how="left_anti"
    )

    .withColumn(
        "motivo_rejeicao",
        F.lit(
            "Venda correspondente não encontrada na camada Silver"
        )
    )

    .withColumn(
        "_data_processamento_quarentena",
        F.current_timestamp()
    )
)

In [0]:
salvar_silver(
    df_quarentena_contas_receber,
    "quarentena_contas_receber"
)

In [0]:
display(
    df_quarentena_contas_receber
)

In [0]:
df_fato_contas_receber_validas = (

    df_fato_contas_receber.alias("r")

    .join(
        df_vendas_validas.alias("v"),

        on="id_venda",

        how="left_semi"
    )
)

In [0]:
salvar_silver(
    df_fato_contas_receber_validas,
    "fato_contas_receber"
)

In [0]:
df_fato_contas_receber = (
    df_fato_contas_receber_validas
)

In [0]:
# ---------------------------------------------------------
# FATO CONTAS A PAGAR
# ---------------------------------------------------------

df_pagar_bronze = carregar_bronze(
    "contas_pagar"
)


df_fato_contas_pagar = (

    df_pagar_bronze

    .select(

        F.upper(
            F.trim(
                F.col(
                    "id_titulo_pagar"
                )
            )
        ).alias(
            "id_titulo_pagar"
        ),

        F.upper(
            F.trim(
                F.col("id_compra")
            )
        ).alias(
            "id_compra"
        ),

        F.upper(
            F.trim(
                F.col("id_fornecedor")
            )
        ).alias(
            "id_fornecedor"
        ),

        F.to_date(
            F.col("data_emissao")
        ).alias(
            "data_emissao"
        ),

        F.to_date(
            F.col("data_vencimento")
        ).alias(
            "data_vencimento"
        ),

        F.to_date(
            F.col("data_pagamento")
        ).alias(
            "data_pagamento"
        ),

        F.col(
            "valor_titulo"
        )
        .cast(
            "decimal(18,2)"
        )
        .alias(
            "valor_titulo"
        ),

        F.trim(
            F.col("status_titulo")
        ).alias(
            "status_titulo"
        ),

        F.col(
            "_data_ingestao"
        ).alias(
            "_data_ingestao_origem"
        )
    )

    .dropDuplicates(
        ["id_titulo_pagar"]
    )

    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)


salvar_silver(
    df_fato_contas_pagar,
    "fato_contas_pagar"
)

In [0]:
# ---------------------------------------------------------
# FATO METAS DE VENDAS
# ---------------------------------------------------------

df_metas_bronze = carregar_bronze(
    "metas_vendas"
)


df_fato_metas_vendas = (

    df_metas_bronze

    .select(

        F.to_date(
            F.concat(
                F.col("ano_mes"),
                F.lit("-01")
            )
        ).alias(
            "mes_referencia"
        ),

        F.upper(
            F.trim(
                F.col("id_vendedor")
            )
        ).alias(
            "id_vendedor"
        ),

        F.col(
            "valor_meta"
        )
        .cast(
            "decimal(18,2)"
        )
        .alias(
            "valor_meta"
        ),

        F.col(
            "meta_clientes"
        )
        .cast("int")
        .alias(
            "meta_clientes"
        ),

        F.col(
            "_data_ingestao"
        ).alias(
            "_data_ingestao_origem"
        )
    )

    .dropDuplicates(
        [
            "mes_referencia",
            "id_vendedor"
        ]
    )

    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)


salvar_silver(
    df_fato_metas_vendas,
    "fato_metas_vendas"
)

In [0]:
display(
    spark.sql(
        f"""
        SHOW TABLES
        IN `{catalogo_atual}`.`{schema_silver}`
        """
    )
)

In [0]:
display(
    df_dim_cliente

    .groupBy(
        "id_cliente"
    )

    .count()

    .filter(
        F.col("count") > 1
    )
)

In [0]:
display(
    df_fato_vendas

    .groupBy(
        "id_item_venda"
    )

    .count()

    .filter(
        F.col("count") > 1
    )
)

In [0]:
display(
    df_dim_cliente

    .groupBy(
        "estado"
    )

    .count()

    .orderBy(
        "estado"
    )
)

In [0]:
display(
    df_dim_cliente

    .groupBy(
        "segmento_cliente"
    )

    .count()

    .orderBy(
        F.desc("count")
    )
)

In [0]:
display(
    df_fato_contas_receber

    .groupBy(
        "status_titulo"
    )

    .count()

    .orderBy(
        F.desc("count")
    )
)

In [0]:
display(
    df_quarentena_vendas

    .groupBy(
        "motivo_rejeicao"
    )

    .count()

    .orderBy(
        F.desc("count")
    )
)

In [0]:
comparacao = []


pares = [

    (
        "clientes",
        "dim_cliente"
    ),

    (
        "produtos",
        "dim_produto"
    ),

    (
        "vendedores",
        "dim_vendedor"
    ),

    (
        "fornecedores",
        "dim_fornecedor"
    ),

    (
        "vendas",
        "fato_vendas"
    ),

    (
        "compras",
        "fato_compras"
    ),

    (
        "estoque",
        "fato_estoque"
    ),

    (
        "contas_receber",
        "fato_contas_receber"
    ),

    (
        "contas_pagar",
        "fato_contas_pagar"
    ),

    (
        "metas_vendas",
        "fato_metas_vendas"
    )
]


for bronze, silver in pares:

    qtd_bronze = (
        spark.table(
            f"{catalogo_atual}."
            f"{schema_bronze}."
            f"{bronze}"
        )
        .count()
    )


    qtd_silver = (
        spark.table(
            f"{catalogo_atual}."
            f"{schema_silver}."
            f"{silver}"
        )
        .count()
    )


    comparacao.append(
        (
            bronze,
            qtd_bronze,
            qtd_silver,
            qtd_bronze - qtd_silver
        )
    )


df_comparacao = spark.createDataFrame(

    comparacao,

    [
        "fonte",
        "registros_bronze",
        "registros_silver",
        "diferenca"
    ]
)


display(
    df_comparacao
)